# Example 4 — Optical cross-talk

Cross-talk is measured per 2×2 macropixel from two on-chip counters, read over
the same frames:

- **`coinc`**: frames in which both micropixels of a pair fired;
- **`OR`**: frames in which at least one of them fired.

```
CT = sum(coinc) / sum(OR)
```

The ratio is dimensionless, so no exposure time is needed. It slightly
overestimates the true optical cross-talk, because independent dark counts in
the same frame also count as coincidences.

Full explanation: [Cross-talk](../docs/guide/crosstalk.md).

## The SPAD indices run clockwise

```
S0 (0,0)   S1 (0,1)
S3 (1,0)   S2 (1,1)
```

So `S0S1` is **horizontal**, `S0S3` **vertical** and `S0S2` **diagonal**. That is
the reverse of a row-major reading.

## What you need

One folder per pairing, each holding many coinc/OR acquisitions:

```
D:\Kelpie\crosstalk\
    S0S1\   CT_coincS0S1_0.bin  CT_ORS0S1_0.bin  CT_coincS0S1_1.bin ...
    S0S2\   CT_coincS0S2_0.bin  CT_ORS0S2_0.bin  ...
    S0S3\   ...
```

- The pairing is read from the file names.
- `coinc` and `OR` are told apart by those tokens, case-sensitively.
- The default `tag="CT_"` selects the main batch. Preliminary `CT01_` / `CT02_`
  batches are never mixed in.

You also need `nframes`, the frames per file you acquired with, one value for
every pairing.

## 1. One pairing

In [1]:
import glob
import os

from IPython.display import Image, display


def show_figures(folder, kind):
    """Show the figures plot_* saved to results/<kind>/ (it closes them)."""
    for png in sorted(glob.glob(os.path.join(folder, "results", kind, "*.png"))):
        display(Image(filename=png))

In [ ]:
import numpy as np

from dapkel.functions import crosstalk_analysis as ct

parent = r"D:\Kelpie\2026.08.07\22V"
nframes = 8192

path = parent + r"\S0S1"
ct.process_crosstalk(path, nframes)  # -> processed/S0S1_ct_crosstalk.npy
ct_map = ct.plot_crosstalk(path)  # heatmap + sorted distribution, in %
show_figures(path, "crosstalk")


> > > Combining per-direction S0 cross-talk into a 64x64 map < < <



FileNotFoundError: [WinError 3] The system cannot find the path specified: 'D:\\Kelpie\\crosstalk\\S0S1'

The returned map is a ratio, not a percentage. Macropixels where `OR = 0` are
NaN, so use NaN-aware statistics:

In [ ]:
print(
    f"S0S1 median {np.nanmedian(ct_map) * 100:.2f} %, "
    f"max {np.nanmax(ct_map) * 100:.2f} %, "
    f"{np.isnan(ct_map).sum()} undefined macropixels"
)

## 2. All directions at once

Point the same call at the **parent** folder. It does two things:

1. It processes and saves every pairing sub-folder, exactly as in step 1.
2. It saves a combined `(64, 64)` micropixel map in the parent. In each 2×2
   block, every neighbour cell holds S0's cross-talk in that direction, and the
   S0 cell holds the mean over the measured directions:

```
mean     | horizontal
vertical | diagonal
```

In [ ]:
ct.process_crosstalk(parent, nframes)
combined = ct.plot_crosstalk(parent)  # 64x64 heatmap + per-direction bar chart
show_figures(parent, "crosstalk")

If a direction is missing, the printout reports it. Sub-folders that are not
S0-based pairings are skipped.

## 3. Per-direction numbers from the combined map

In [ ]:
from dapkel.core import store

combined, meta = store.load_map(parent, kind="crosstalk", tag="CT_")
print(meta["median_ct_percent"])  # {'diagonal': ..., 'horizontal': ..., ...}

horizontal = combined[0::2, 1::2]  # (32, 32), S0 -> S1
vertical = combined[1::2, 0::2]  # S0 -> S3
diagonal = combined[1::2, 1::2]  # S0 -> S2
for name, m in [
    ("horizontal", horizontal),
    ("vertical", vertical),
    ("diagonal", diagonal),
]:
    print(f"{name:10s} median {np.nanmedian(m) * 100:.2f} %")

Diagonal SPADs sit further apart than horizontal or vertical ones, so diagonal
cross-talk should come out lowest. If it does not, check the folder names first.